In [ ]:
# Phase 4: Batch Model Scoring - Equipment Stop Prediction
# Objective: Load trained models and score latest sensor data from Eventhouse real-time feeds
# Sources: PI, iCare, GADS data from pi-realtime-db Eventhouse
# Output: ml.predictions_shortterm (asset_id, timestamp, horizon, probability, alert_level)

from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime, timedelta
import mlflow

# Eventhouse connection config
KUSTO_URI = "https://trd-8a08ckb2duw406mvvg.z2.kusto.fabric.microsoft.com"
KUSTO_DB = "pi-realtime-db"

def _kusto_tok():
    try:
        import notebookutils as _n; _c = _n.credentials
    except Exception:
        from notebookutils import mssparkutils as _m; _c = _m.credentials
    for _a in (KUSTO_URI, "kusto", "pbi"):
        try:
            _t = _c.getToken(_a)
            if _t: return _t
        except Exception:
            pass
    raise RuntimeError("could not acquire Kusto token")

def read_kusto(query):
    """Read from Eventhouse KQL database via Kusto Spark connector"""
    return (spark.read
        .format("com.microsoft.kusto.spark.datasource")
        .option("accessToken", _kusto_tok())
        .option("kustoCluster", KUSTO_URI)
        .option("kustoDatabase", KUSTO_DB)
        .option("kustoQuery", query)
        .load())

print("Phase 4 Batch Scoring initialized")
print(f"Scoring run timestamp: {datetime.now()}")
print(f"Eventhouse: {KUSTO_URI} / {KUSTO_DB}")

## Step 1 - Load Best Models from MLflow
Retrieve the production models trained in Phase 3 (GradientBoosting and XGBoost winners per horizon).

In [ ]:
# Load best stop and derate models from Phase 3 MLflow experiment
EXPERIMENT_NAME = "Phase3-Predictive-Model"
LABEL_TYPES = ['stop', 'derate']
HORIZONS = ['4h', '8h', '24h']

models = {}
model_info = {}

try:
    for label_type in LABEL_TYPES:
        models[label_type] = {}
        model_info[label_type] = {}

        for horizon in HORIZONS:
            # Search for best model by ROC AUC - no n_features constraint
            runs_df = mlflow.search_runs(
                experiment_names=[EXPERIMENT_NAME],
                filter_string=f"params.horizon = '{horizon}' and params.label_type = '{label_type}'",
                order_by=["metrics.training_roc_auc DESC"]
            )

            if len(runs_df) == 0:
                print(f"  No runs found for {label_type}/{horizon} - will use fallback method")
                raise ValueError("Required model params not found, using fallback")

            best_run = runs_df.iloc[0]
            model_uri = f"runs:/{best_run.run_id}/model"
            n_feats = best_run.get('params.n_features', '?')

            try:
                models[label_type][horizon] = mlflow.sklearn.load_model(model_uri)
            except:
                models[label_type][horizon] = mlflow.xgboost.load_model(model_uri)

            model_info[label_type][horizon] = {
                'run_id': best_run.run_id,
                'roc_auc': best_run['metrics.training_roc_auc'],
                'algorithm': best_run.get('params.algorithm', 'Unknown'),
                'n_features': n_feats
            }
            print(f"  {label_type}/{horizon} model loaded: {model_info[label_type][horizon]['algorithm']} "
                  f"(ROC AUC: {model_info[label_type][horizon]['roc_auc']:.3f}, features: {n_feats})")

except Exception as e:
    print(f"\n  Using fallback: manual run ID mapping (reason: {e})")

    manual_runs = {
        'stop': {
            '4h': 'a45a3772-2426-4a6e-940a-80f76842716d',
            '8h': '51a82d59-cb76-48ee-bc2a-cc14f1aa8a1f',
            '24h': 'cc1be92d-023a-405e-8183-fe0e3da734b0'
        },
        'derate': {
            '4h': 'YOUR_DERATE_4H_RUN_ID',
            '8h': 'YOUR_DERATE_8H_RUN_ID',
            '24h': 'YOUR_DERATE_24H_RUN_ID'
        }
    }

    for label_type, horizon_runs in manual_runs.items():
        models[label_type] = {}
        model_info[label_type] = {}

        for horizon, run_id in horizon_runs.items():
            if run_id.startswith('YOUR_'):
                print(f"  Skipping {label_type}/{horizon} - no run ID configured")
                continue

            model_uri = f"runs:/{run_id}/model"
            try:
                models[label_type][horizon] = mlflow.sklearn.load_model(model_uri)
            except:
                models[label_type][horizon] = mlflow.xgboost.load_model(model_uri)

            model_info[label_type][horizon] = {'run_id': run_id, 'roc_auc': None, 'algorithm': 'Unknown', 'n_features': '?'}
            print(f"  {label_type}/{horizon} model loaded from run {run_id}")

print("\n[OK] All available models loaded")

In [ ]:
# Load Phase 2 selected tags (top correlated features used in training)
# These tags are asset-specific - each asset has its own set of predictive tags
selected_tags_df = spark.table("ml.selected_tags")

# Check actual column names in the table
print(f"Table columns: {selected_tags_df.columns}")

# Build asset-to-tags mapping (each asset has different tags)
asset_tags = {}
for row in selected_tags_df.collect():
    asset_id = row.tag_asset_id if "tag_asset_id" in selected_tags_df.columns else row.asset_id
    tag = row.Tag if "Tag" in selected_tags_df.columns else row.tag_name
    
    if asset_id not in asset_tags:
        asset_tags[asset_id] = []
    asset_tags[asset_id].append(tag)

# Remap short asset_ids to canonical dim_asset names
ASSET_REMAP = {
    "RV3_U3": "RV3_U3_Steam_Turbine",

}
asset_tags = {ASSET_REMAP.get(k, k): v for k, v in asset_tags.items()}


for asset_id, tags in asset_tags.items():
    print(f"  {asset_id}: {len(tags)} tags")
    
# Get all unique tags for Eventhouse query (union of all asset tags)
all_selected_tags = list(set([tag for tags in asset_tags.values() for tag in tags]))
print(f"\nTotal unique tags to query: {len(all_selected_tags)}")

## Step 2 - Extract Latest Sensor Data from Eventhouse
Pull the most recent 4 hours of PI and iCare data from the real-time Eventhouse feeds (PiEvents, IcareEvents).

In [ ]:
# Configuration
TARGET_ASSETS = list(asset_tags.keys())  # Use remapped asset names from Cell 3
SCORING_WINDOW_HOURS = 26  # Need 24h+ of history for rolling features (24h window + 2h buffer)

BIN_SIZE_MINUTES = 15

scoring_time = datetime.now()
start_time = scoring_time - timedelta(hours=SCORING_WINDOW_HOURS)

print(f"Scoring window: {start_time} to {scoring_time}")
print(f"Target assets: {TARGET_ASSETS}")

# Build KQL-compatible tag filter string (all selected tags from Phase 2)
kql_tag_list = "'" + "','".join(all_selected_tags) + "'"
print(f"\Filtering Eventhouse query to {len(all_selected_tags)} selected tags")

# --- Pull PI data from Eventhouse (PiEvents table) - FILTERED TO SELECTED TAGS ---
try:
    pi_data = read_kusto(f"""
        PiEvents
        | where Ts >= datetime({start_time.strftime('%Y-%m-%d %H:%M:%S')})
          and Ts <= datetime({scoring_time.strftime('%Y-%m-%d %H:%M:%S')})
          and not(Questionable)
          and Tag in ({kql_tag_list})
        | project tag_name = Tag, timestamp = Ts, value = toreal(Value)
    """)
    pi_count = pi_data.count()
    print(f"\nPI data (Eventhouse, filtered): {pi_count:,} rows")
except Exception as e:
    print(f"\nPI data: Error - {e}")
    pi_data = spark.createDataFrame([], "tag_name STRING, timestamp TIMESTAMP, value DOUBLE")
    pi_count = 0

# --- Pull iCare data from Eventhouse (IcareEvents table) ---
try:
    icare_data = read_kusto(f"""
        IcareEvents
        | where acqend >= datetime({start_time.strftime('%Y-%m-%d %H:%M:%S')})
          and acqend <= datetime({scoring_time.strftime('%Y-%m-%d %H:%M:%S')})
        | project tag_name = strcat('ICARE:', global_type), timestamp = acqend, value = toreal(value)
    """)
    icare_count = icare_data.count()
    print(f"iCare data (Eventhouse): {icare_count:,} rows")
except Exception as e:
    print(f"iCare data: No records yet (expected) - {e}")
    icare_data = spark.createDataFrame([], "tag_name STRING, timestamp TIMESTAMP, value DOUBLE")
    icare_count = 0

# --- Union all sensor data ---
all_sensor_data = pi_data.unionByName(icare_data)
total_count = pi_count + icare_count
print(f"\nTotal sensor readings: {total_count:,}")

if total_count == 0:
    print("\nWARNING: No sensor data available. PI forwarder may not be running.")
    print("Scoring will continue but predictions will be empty.")

## Step 3 - Apply Phase 2 Feature Engineering
Reuse the exact same feature engineering logic from Phase 2 to ensure consistency.

## Diagnostic: Feature Column Alignment Check
Compare training feature columns (from Phase 2) with Phase 4's generated columns to identify the mismatch.

In [ ]:
# Get training feature order from the LOADED MODEL (not from training table)
# This ensures Phase 4 features match exactly what the model expects

sample_model = None
for label_type in models:
    for horizon in models[label_type]:
        sample_model = models[label_type][horizon]
        break
    if sample_model:
        break

if sample_model is None:
    raise ValueError("No models loaded - cannot determine feature order")

expected_features = sample_model.n_features_in_
print(f"Models expect {expected_features} features")

if hasattr(sample_model, 'feature_names_in_'):
    training_feature_order = list(sample_model.feature_names_in_)
    print(f"[OK] Got {len(training_feature_order)} feature names from model.feature_names_in_")
else:
    # Fallback: read from training table, take first N sorted alphabetically
    print("Model lacks feature_names_in_, falling back to training table...")
    training_df = spark.table("ml.training_shortterm")
    exclude_cols = ['asset_id', 'timestamp_bin', 'hours_to_next_stop', 'hours_to_next_derate',
                    'label_stop_4h', 'label_stop_8h', 'label_stop_24h',
                    'label_derate_4h', 'label_derate_8h', 'label_derate_24h',
                    'is_during_outage', 'event_type_4h', 'event_type_8h', 'event_type_24h']
    all_features = sorted([c for c in training_df.columns if c not in exclude_cols])
    if len(all_features) == expected_features:
        training_feature_order = all_features
        print(f"[OK] Training table has exactly {expected_features} features - using all")
    elif len(all_features) > expected_features:
        training_feature_order = all_features[:expected_features]
        print(f"[WARN] Training table has {len(all_features)} features, model expects {expected_features}")
        print(f"  Using first {expected_features} alphabetically - recommend re-running Phase 3")
    else:
        training_feature_order = all_features
        print(f"[WARN] Training table has fewer features ({len(all_features)}) than model expects ({expected_features})")

# Show derivation types in the feature set
import re
suffixes = set()
for f in training_feature_order:
    m = re.search(r'__(\w+)$', f)
    if m:
        suffixes.add(m.group(1))

print(f"\nFeature schema: {len(training_feature_order)} columns")
print(f"Derivation types: {sorted(suffixes)}")
print(f"Sample: {training_feature_order[:5]}")

In [ ]:
# FIXED: Feature engineering aligned to training schema
# Key change: Build features per asset, then map into the EXACT training column order

if total_count == 0:
    print("No sensor data - skipping feature engineering")
    scoring_features_df = None
else:
    import pandas as pd
    import numpy as np
    from pyspark.sql.functions import window, avg, col

    # Step 0: Get exact training feature order (set in diagnostic cell above)
    # training_feature_order was defined in the diagnostic cell
    print(f"Training schema: {len(training_feature_order)} feature columns")
    
    asset_aligned_rows = []  # Collect aligned feature rows per asset
    
    for asset_id in TARGET_ASSETS:
        print(f"\n--- Processing {asset_id} ---")
        
        # Get this asset's specific tags
        asset_tag_list = asset_tags.get(asset_id, [])
        if len(asset_tag_list) == 0:
            print(f"   No tags found for {asset_id} - skipping")
            continue
            
        print(f"  Tags for this asset: {len(asset_tag_list)}")
        
        # Filter sensor data to only this asset's tags
        asset_sensor_data = all_sensor_data.filter(col("tag_name").isin(asset_tag_list))
        asset_row_count = asset_sensor_data.count()
        print(f"  Sensor readings: {asset_row_count:,}")
        
        if asset_row_count == 0:
            print(f"   No data for {asset_id} - skipping")
            continue
        
        # Step 1: Bin to 15-min intervals in Spark
        binned_data = asset_sensor_data.groupBy(
            window(col("timestamp"), f"{BIN_SIZE_MINUTES} minutes").alias("time_window"),
            col("tag_name")
        ).agg(
            avg("value").alias("value")
        ).select(
            col("time_window.start").alias("timestamp_bin"),
            col("tag_name"),
            col("value")
        )

        # Step 2: Collect to Pandas
        binned_pdf = binned_data.toPandas()
        print(f"  Binned data: {len(binned_pdf):,} rows")

        # Step 3: Pivot in Pandas
        pivoted = binned_pdf.pivot_table(index='timestamp_bin', columns='tag_name', values='value')
        pivoted = pivoted.sort_index()
        print(f"  Pivoted: {pivoted.shape[0]} timestamps x {pivoted.shape[1]} tags")

        # Step 4: Build derived features (same logic as Phase 2 v4 - 8 derivations)
        asset_feature_dict = {}
        for tag_col in pivoted.columns:
            import re
            safe_tag = re.sub(r'[^a-zA-Z0-9_]', '_', tag_col)
            asset_feature_dict[f"{safe_tag}__v"] = pivoted[tag_col].values
            asset_feature_dict[f"{safe_tag}__avg1h"] = pivoted[tag_col].rolling(4, min_periods=1).mean().values
            asset_feature_dict[f"{safe_tag}__delta1h"] = (pivoted[tag_col] - pivoted[tag_col].shift(4)).values
            asset_feature_dict[f"{safe_tag}__avg4h"] = pivoted[tag_col].rolling(16, min_periods=1).mean().values
            asset_feature_dict[f"{safe_tag}__delta4h"] = (pivoted[tag_col] - pivoted[tag_col].shift(16)).values
            asset_feature_dict[f"{safe_tag}__avg8h"] = pivoted[tag_col].rolling(32, min_periods=1).mean().values
            asset_feature_dict[f"{safe_tag}__delta8h"] = (pivoted[tag_col] - pivoted[tag_col].shift(32)).values
            asset_feature_dict[f"{safe_tag}__std1h"] = pivoted[tag_col].rolling(4, min_periods=1).std().values
            asset_feature_dict[f"{safe_tag}__std4h"] = pivoted[tag_col].rolling(16, min_periods=1).std().values
            asset_feature_dict[f"{safe_tag}__avg12h"] = pivoted[tag_col].rolling(48, min_periods=1).mean().values
            asset_feature_dict[f"{safe_tag}__delta12h"] = (pivoted[tag_col] - pivoted[tag_col].shift(48)).values
            asset_feature_dict[f"{safe_tag}__avg24h"] = pivoted[tag_col].rolling(96, min_periods=1).mean().values
            asset_feature_dict[f"{safe_tag}__delta24h"] = (pivoted[tag_col] - pivoted[tag_col].shift(96)).values
        
        asset_feature_names = list(asset_feature_dict.keys())
        
        # Step 5: CRITICAL - Map into training column order
        # Create a full-width row with ALL training columns, zeros for other assets' features
        for row_idx in range(len(pivoted)):
            aligned_row = np.zeros(len(training_feature_order))
            
            matched = 0
            for i, train_col in enumerate(training_feature_order):
                if train_col in asset_feature_dict:
                    val = asset_feature_dict[train_col][row_idx]
                    aligned_row[i] = val if not pd.isna(val) else 0.0
                    matched += 1
            
            asset_aligned_rows.append({
                'asset_id': asset_id,
                'timestamp_bin': pivoted.index[row_idx],
                'features': aligned_row,
                'matched_features': matched
            })
        
        # Report alignment quality
        in_training = sum(1 for f in asset_feature_names if f in training_feature_order)
        not_in_training = sum(1 for f in asset_feature_names if f not in training_feature_order)
        print(f"   Generated {len(asset_feature_names)} features")
        print(f"   Matched to training schema: {in_training}/{len(asset_feature_names)}")
        if not_in_training > 0:
            missing = [f for f in asset_feature_names if f not in training_feature_order]
            print(f"   {not_in_training} features NOT in training schema (new tags?): {missing[:5]}")
        print(f"   Rows generated: {len(pivoted)}")
    
    if len(asset_aligned_rows) > 0:
        # Build a DataFrame with correctly aligned features
        scoring_features_df = pd.DataFrame({
            'asset_id': [r['asset_id'] for r in asset_aligned_rows],
            'timestamp_bin': [r['timestamp_bin'] for r in asset_aligned_rows]
        })
        
        # Add all feature columns in correct order
        feature_matrix = np.vstack([r['features'] for r in asset_aligned_rows])
        for i, col_name in enumerate(training_feature_order):
            scoring_features_df[col_name] = feature_matrix[:, i]
        
        print(f" Feature engineering complete")
        print(f"  Total rows: {len(scoring_features_df)}")
        print(f"  Feature columns: {len(training_feature_order)} (exactly matches training)")
        print(f"  Assets: {scoring_features_df['asset_id'].nunique()}")
        
        # Verify alignment
        for asset in scoring_features_df['asset_id'].unique():
            asset_row = scoring_features_df[scoring_features_df['asset_id'] == asset].iloc[0]
            nonzero = (asset_row[training_feature_order] != 0).sum()
            print(f"  {asset}: {nonzero}/{len(training_feature_order)} non-zero features")
    else:
        scoring_features_df = None
        print("No features generated")

## Step 4 - Score Latest State
Apply models to the most recent timestamp to get stop probabilities for each horizon.

In [ ]:
# FIXED: Score using correctly aligned features
# No more pad/trim - features are in exact training column order

if scoring_features_df is None or len(scoring_features_df) == 0:
    print("No features available - skipping scoring")
    predictions = {}
    scoring_timestamp = datetime.now()
else:
    import numpy as np

    loaded_models = [model for label_models in models.values() for model in label_models.values()]
    if len(loaded_models) == 0:
        raise ValueError("No models available for scoring")

    predictions = {}
    scoring_timestamp = None

    for asset_id in scoring_features_df['asset_id'].unique():
        print(f"\n--- Scoring {asset_id} ---")

        asset_rows = scoring_features_df[scoring_features_df['asset_id'] == asset_id].copy()
        asset_rows = asset_rows.sort_values('timestamp_bin', ascending=False)
        latest = asset_rows.head(1)

        asset_scoring_timestamp = latest['timestamp_bin'].iloc[0]
        if scoring_timestamp is None:
            scoring_timestamp = asset_scoring_timestamp

        print(f"  Scoring timestamp: {asset_scoring_timestamp}")

        X_score = latest[training_feature_order].values

        n_features = X_score.shape[1]
        n_nonzero = (X_score[0] != 0).sum()
        expected_features = loaded_models[0].n_features_in_
        print(f"  Features: {n_features} (training expects: {expected_features})")
        print(f"  Non-zero features: {n_nonzero} (this asset's sensor values)")

        if n_features != expected_features:
            print(f"  FATAL: Feature count {n_features} != model expects {expected_features}")
            print(f"     Training schema may have changed. Re-run Phase 2 & 3.")
            continue

        asset_predictions = {}
        for label_type, label_models in models.items():
            asset_predictions[label_type] = {}

            for horizon, model in label_models.items():
                prob_event = model.predict_proba(X_score)[0, 1]

                if prob_event >= 0.7:
                    alert_level = 'HIGH'
                elif prob_event >= 0.4:
                    alert_level = 'MEDIUM'
                elif prob_event >= 0.2:
                    alert_level = 'LOW'
                else:
                    alert_level = 'NORMAL'

                asset_predictions[label_type][horizon] = {
                    'probability': float(prob_event),
                    'alert_level': alert_level,
                    'model_algorithm': model_info[label_type][horizon]['algorithm'],
                    'model_run_id': model_info[label_type][horizon]['run_id'],
                    'timestamp': asset_scoring_timestamp
                }

                print(f"  {label_type} {horizon}: {prob_event:.3f} ({alert_level})")

        predictions[asset_id] = asset_predictions

    print(f"\n Scoring complete - {len(predictions)} assets scored")
    print(f"  Features correctly aligned to training schema ")
    print(f"  Stop and derate predictions generated")

## Step 5 - Save Predictions to Gold Schema
Write predictions to `ml.predictions_shortterm` table for downstream consumption (Power BI, Activator alerts).

In [ ]:
import pandas as pd

if not predictions:
    print("No predictions to save (no sensor data available)")
    pred_rows = []
else:
    pred_rows = []
    for asset_id, asset_label_predictions in predictions.items():
        for label_type, asset_preds in asset_label_predictions.items():
            for horizon, pred in asset_preds.items():
                pred_rows.append({
                    'scoring_timestamp': pred['timestamp'],
                    'asset_id': asset_id,
                    'prediction_horizon': horizon,
                    'label_type': label_type,
                    'stop_probability': pred['probability'],
                    'alert_level': pred['alert_level'],
                    'model_algorithm': pred['model_algorithm'],
                    'model_run_id': pred['model_run_id'],
                    'scored_at': datetime.now()
                })

    pred_df = spark.createDataFrame(pd.DataFrame(pred_rows))
    pred_df.write.mode("append").option("mergeSchema", "true").option("overwriteSchema", "true").saveAsTable("ml.predictions_shortterm")
    print(f"Saved {len(pred_rows)} predictions to ml.predictions_shortterm")
    pred_df.show(truncate=False)

## SHAP Explainability: Why Did the Model Predict This?
Use SHAP (SHapley Additive exPlanations) to show which sensor features drove each prediction up or down.

In [ ]:
# SHAP Explainability + Persist to ml.drivers_shortterm
# For each prediction: compute SHAP, print summary, save top contributors to Delta table

import shap
import numpy as np
import pandas as pd

TOP_N = 10  # Top N features per prediction to save

shap_rows = []  # Collect rows for gold table

for asset_id, asset_label_predictions in predictions.items():
    print(f"\n{'='*90}")
    print(f"SHAP EXPLANATION: {asset_id}")
    print(f"{'='*90}")

    asset_rows = scoring_features_df[scoring_features_df['asset_id'] == asset_id].copy()
    asset_rows = asset_rows.sort_values('timestamp_bin', ascending=False)
    latest = asset_rows.head(1)
    X_single = latest[training_feature_order].values

    for label_type, asset_preds in asset_label_predictions.items():
        for horizon, pred in asset_preds.items():
            model = models[label_type][horizon]
            prob = pred['probability']
            alert = pred['alert_level']

            print(f"\n--- {label_type} {horizon} Prediction: {prob:.3f} ({alert}) ---")

            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(X_single)

            if isinstance(shap_values, list):
                sv = shap_values[1][0]
            else:
                sv = shap_values[0]

            explanation = pd.DataFrame({
                'feature': training_feature_order,
                'shap_value': sv,
                'feature_value': X_single[0],
                'abs_shap': np.abs(sv)
            }).sort_values('abs_shap', ascending=False)

            explanation['base_tag'] = explanation['feature'].str.replace(
                r'__(v|avg\d+h|delta\d+h|std\d+h)$', '', regex=True)
            explanation['derivation'] = explanation['feature'].str.extract(
                r'__(v|avg\d+h|delta\d+h|std\d+h)$')[0].fillna('raw')

            pushing_up = explanation[explanation['shap_value'] > 0].head(TOP_N)
            if len(pushing_up) > 0:
                print(f"\n Top features INCREASING {label_type} probability:")
                for _, row in pushing_up.iterrows():
                    val_str = f"value={row['feature_value']:.2f}" if row['feature_value'] != 0 else "value=0"
                    print(f"     {row['shap_value']:+.4f}  {row['feature']:50s} ({val_str})")

            pushing_down = explanation[explanation['shap_value'] < 0].sort_values('shap_value').head(TOP_N)
            if len(pushing_down) > 0:
                print(f"\n  Top features DECREASING {label_type} probability:")
                for _, row in pushing_down.iterrows():
                    val_str = f"value={row['feature_value']:.2f}" if row['feature_value'] != 0 else "value=0"
                    print(f"     {row['shap_value']:+.4f}  {row['feature']:50s} ({val_str})")

            tag_shap = explanation.groupby('base_tag')['shap_value'].sum().sort_values(key=abs, ascending=False)
            print(f"\n Top 5 most influential SENSORS:")
            for tag, total_shap in tag_shap.head(5).items():
                direction = f"{label_type.upper()}" if total_shap > 0 else "NORMAL"
                print(f"     {total_shap:+.4f}  {tag:50s} ({direction})")

            top_features = pd.concat([pushing_up.head(TOP_N), pushing_down.head(TOP_N)])
            for _, row in top_features.iterrows():
                shap_rows.append({
                    'scoring_timestamp': pred['timestamp'],
                    'scored_at': datetime.now(),
                    'asset_id': asset_id,
                    'prediction_horizon': horizon,
                    'label_type': label_type,
                    'stop_probability': prob,
                    'alert_level': alert,
                    'feature_name': row['feature'],
                    'base_tag': row['base_tag'],
                    'derivation': row['derivation'],
                    'shap_value': float(row['shap_value']),
                    'feature_value': float(row['feature_value']),
                    'contribution_direction': 'INCREASING' if row['shap_value'] > 0 else 'DECREASING',
                    'feature_rank': int(row.name) if hasattr(row, 'name') else 0,
                    'model_run_id': model_info[label_type][horizon]['run_id']
                })

# =============================================================================
# Historical pattern-awareness for the surfaced drivers
# Compare each INCREASING driver's CURRENT value to how that sensor behaves during
# NORMAL operation (long history in ml.training_shortterm, excluding stop/derate
# pre-event bins). A strong driver whose value is still inside its normal p5-p95
# band is a recurring pattern (NORMAL_PATTERN); a value outside that band is a
# genuine short-term deviation (DEVIATION). The watchlist cell uses this to
# downgrade normal patterns and quantify real deviations - no post-hoc cleanup.
# =============================================================================
from pyspark.sql import functions as _F

_needed = {}
for _r in shap_rows:
    if _r.get('contribution_direction') == 'INCREASING' and _r.get('alert_level') in ('MEDIUM', 'HIGH'):
        _needed.setdefault(_r['asset_id'], set()).add(_r['feature_name'])

driver_baselines = {}
if _needed:
    _ts = spark.table("ml.training_shortterm")
    _tscols = set(_ts.columns)
    _feats = sorted({f for s in _needed.values() for f in s if f in _tscols})
    if _feats:
        _norm = _ts
        for _lbl in ("label_stop_8h", "label_derate_8h"):
            if _lbl in _tscols:
                _norm = _norm.filter(_F.col(_lbl) == 0)
        _colmap = {}
        _aggs = []
        for _i, _f in enumerate(_feats):
            _colmap[_f] = _i
            _aggs += [
                _F.expr("percentile_approx(`%s`, 0.05)" % _f).alias("c%d_p05" % _i),
                _F.expr("percentile_approx(`%s`, 0.50)" % _f).alias("c%d_p50" % _i),
                _F.expr("percentile_approx(`%s`, 0.95)" % _f).alias("c%d_p95" % _i),
                _F.count(_F.col("`%s`" % _f)).alias("c%d_n" % _i),
            ]
        _stats = _norm.groupBy("asset_id").agg(*_aggs).toPandas().set_index("asset_id")
        for _a, _s in _needed.items():
            if _a not in _stats.index:
                continue
            _row = _stats.loc[_a]
            for _f in _s:
                if _f not in _colmap:
                    continue
                _i = _colmap[_f]
                _n = _row.get("c%d_n" % _i)
                _p50 = _row.get("c%d_p50" % _i)
                if _n is None or _n < 200 or _p50 is None or pd.isna(_p50):
                    continue
                _p05 = float(_row.get("c%d_p05" % _i))
                _p95 = float(_row.get("c%d_p95" % _i))
                _spread = _p95 - _p05
                _std = _spread / 3.29 if _spread and not pd.isna(_spread) else 0.0
                driver_baselines[(_a, _f)] = {'p05': _p05, 'p50': float(_p50), 'p95': _p95, 'std': _std, 'n': int(_n)}

for _r in shap_rows:
    _b = driver_baselines.get((_r['asset_id'], _r['feature_name']))
    if _b and _r.get('contribution_direction') == 'INCREASING':
        _cur = _r['feature_value']
        _within = _b['p05'] <= _cur <= _b['p95']
        _z = (_cur - _b['p50']) / _b['std'] if _b['std'] > 1e-9 else (0.0 if abs(_cur - _b['p50']) < 1e-9 else 99.0)
        _r['driver_value'] = float(_cur)
        _r['hist_median'] = _b['p50']
        _r['hist_low'] = _b['p05']
        _r['hist_high'] = _b['p95']
        _r['hist_std'] = _b['std']
        _r['hist_zscore'] = round(float(_z), 2)
        _r['pattern_flag'] = 'NORMAL_PATTERN' if _within else 'DEVIATION'
    else:
        _r['driver_value'] = float(_r['feature_value']) if _r.get('feature_value') is not None else np.nan
        _r['hist_median'] = np.nan
        _r['hist_low'] = np.nan
        _r['hist_high'] = np.nan
        _r['hist_std'] = np.nan
        _r['hist_zscore'] = np.nan
        _r['pattern_flag'] = 'UNKNOWN'

_ptn_normal = sum(1 for _r in shap_rows if _r.get('pattern_flag') == 'NORMAL_PATTERN')
_ptn_dev = sum(1 for _r in shap_rows if _r.get('pattern_flag') == 'DEVIATION')
print("Pattern awareness: %d NORMAL_PATTERN, %d DEVIATION driver-rows (of %d)" % (_ptn_normal, _ptn_dev, len(shap_rows)))


if shap_rows:
    shap_df = spark.createDataFrame(pd.DataFrame(shap_rows))
    shap_df.write.mode("append").option("mergeSchema", "true").option("overwriteSchema", "true").format("delta").saveAsTable("ml.drivers_shortterm")
    print(f"Saved {len(shap_rows)} SHAP explanations to ml.drivers_shortterm")
    shap_df.select("asset_id", "label_type", "prediction_horizon", "feature_name", "shap_value", 
                    "feature_value", "contribution_direction").show(20, truncate=False)
else:
    print("\nNo SHAP rows to save")

In [ ]:
# Write simplified MEDIUM/HIGH alerts to ml.watchlist
# 1 row per asset per horizon. Drivers, baselines and pattern-awareness come
# straight from the SHAP cell above: each surfaced driver is compared to its
# NORMAL-operation history, so a strong-but-normal driver is downgraded and a
# genuine short-term deviation is kept and quantified. No post-hoc cleanup.

import uuid
import re

watchlist_rows = []
run_id = str(uuid.uuid4())

# Load tag descriptors from bridge table
bridge_df = spark.table("gold.bridge_pi_tag_to_asset").select("Tag", "tag_description", "eng_units").toPandas()
tag_desc_map = dict(zip(bridge_df['Tag'], bridge_df['tag_description']))
tag_units_map = dict(zip(bridge_df['Tag'], bridge_df['eng_units']))

def _base_tag(feature_name):
    return re.sub(r'__(v|avg\d+h|delta\d+h|std\d+h)$', '', feature_name)

def _pi_tag(feature_name):
    b = _base_tag(feature_name)
    return b.replace('_', ':', 1).replace('_', '.', 1)   # RV2_BATU2BT21_AG__avg1h -> RV2:BATU2BT21.AG

def get_friendly_name(feature_name):
    desc = tag_desc_map.get(_pi_tag(feature_name))
    return desc.title() if desc else _base_tag(feature_name)

NOTCH = {'HIGH': 'MEDIUM', 'MEDIUM': 'LOW', 'LOW': 'LOW'}

for asset_id, asset_label_predictions in predictions.items():
    for label_type, label_preds in asset_label_predictions.items():
        for horizon, pred in label_preds.items():
            if pred['alert_level'] not in ('MEDIUM', 'HIGH'):
                continue

            # Top 3 INCREASING SHAP drivers (already pattern-tagged by the SHAP cell)
            asset_shap = [r for r in shap_rows
                          if r['asset_id'] == asset_id
                          and r['label_type'] == label_type
                          and r['prediction_horizon'] == horizon
                          and r['contribution_direction'] == 'INCREASING']
            asset_shap.sort(key=lambda x: abs(x['shap_value']), reverse=True)
            top3 = asset_shap[:3]
            if not top3:
                continue

            friendly_names = [get_friendly_name(d['feature_name']) for d in top3]
            unique_names = list(dict.fromkeys(friendly_names))
            watch_list_str = ", ".join(unique_names[:3])

            top_driver = top3[0]
            pi_tag = _pi_tag(top_driver['feature_name'])   # clean tag for any window suffix

            # --- historical pattern awareness (computed in the SHAP cell) ---
            patterns = [d.get('pattern_flag', 'UNKNOWN') for d in top3]
            known = [p for p in patterns if p != 'UNKNOWN']
            top_pattern = top_driver.get('pattern_flag', 'UNKNOWN')
            all_normal = bool(known) and all(p == 'NORMAL_PATTERN' for p in known)

            orig_alert = pred['alert_level']
            action = orig_alert
            base_text = ("Watch %s over the next %s. %s probability: %.0f%%."
                         % (watch_list_str, horizon, label_type.title(), pred['probability'] * 100))
            if all_normal:
                action = NOTCH.get(orig_alert, orig_alert)
                note = (" Historical check: driver value(s) are at NORMAL levels (within the "
                        "p5-p95 band of normal-operation history) - likely a recurring pattern, "
                        "not a short-term anomaly. Downgraded %s->%s." % (orig_alert, action))
            elif top_pattern == 'DEVIATION':
                _z = top_driver.get('hist_zscore')
                _ztxt = ("~%.1f sigma " % _z) if (_z is not None and not (isinstance(_z, float) and _z != _z)) else ""
                note = (" Historical check: %s is ABNORMAL %svs its normal-operation history "
                        "- genuine short-term deviation." % (unique_names[0], _ztxt))
            else:
                note = ""

            def _num(v):
                if v is None:
                    return None
                if isinstance(v, float) and v != v:   # NaN
                    return None
                return float(v)

            watchlist_rows.append({
                'model_name': 'GBM_ShortTerm',
                'scoring_date': str(pred['timestamp'].date()) if hasattr(pred['timestamp'], 'date') else str(pred['timestamp'])[:10],
                'asset_id': asset_id,
                'feature': f"{label_type}_{horizon}",
                'tag_name': pi_tag,
                'descriptor': f"{label_type.upper()} {horizon} alert",
                'engineering_units': tag_units_map.get(pi_tag),
                'current_value': float(pred['probability']),
                'baseline_mean': _num(top_driver.get('hist_median')),
                'baseline_std': _num(top_driver.get('hist_std')),
                'normal_range_low': _num(top_driver.get('hist_low')),
                'normal_range_high': _num(top_driver.get('hist_high')),
                'risk_contribution': float(abs(top_driver['shap_value'])),
                'trend_direction': 'INCREASING',
                'trend_slope_per_day': None,
                'recommended_action': action,
                'recommendation_text': base_text + note,
                'watch_horizon_days': 1,
                'model_run_timestamp': datetime.now(),
                'notebook_run_id': run_id,
                'driver_value': _num(top_driver.get('driver_value')),
                'pattern_flag': top_pattern,
                'hist_zscore': _num(top_driver.get('hist_zscore'))
            })

if watchlist_rows:
    from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
    schema = StructType([
        StructField('model_name', StringType()),
        StructField('scoring_date', StringType()),
        StructField('asset_id', StringType()),
        StructField('feature', StringType()),
        StructField('tag_name', StringType()),
        StructField('descriptor', StringType()),
        StructField('engineering_units', StringType()),
        StructField('current_value', DoubleType()),
        StructField('baseline_mean', DoubleType()),
        StructField('baseline_std', DoubleType()),
        StructField('normal_range_low', DoubleType()),
        StructField('normal_range_high', DoubleType()),
        StructField('risk_contribution', DoubleType()),
        StructField('trend_direction', StringType()),
        StructField('trend_slope_per_day', DoubleType()),
        StructField('recommended_action', StringType()),
        StructField('recommendation_text', StringType()),
        StructField('watch_horizon_days', IntegerType()),
        StructField('model_run_timestamp', TimestampType()),
        StructField('notebook_run_id', StringType()),
        StructField('driver_value', DoubleType()),
        StructField('pattern_flag', StringType()),
        StructField('hist_zscore', DoubleType())
    ])
    wl_df = spark.createDataFrame(pd.DataFrame(watchlist_rows), schema=schema)
    wl_df.write.mode('append').option('mergeSchema', 'true').saveAsTable('ml.watchlist')
    n_down = sum(1 for r in watchlist_rows if r['recommended_action'] != None and r['pattern_flag'] == 'NORMAL_PATTERN')
    print(f"Saved {len(watchlist_rows)} watchlist entries ({n_down} downgraded as normal patterns)")
    wl_df.select('asset_id', 'tag_name', 'pattern_flag', 'recommended_action', 'recommendation_text').show(truncate=False)
else:
    print('No MEDIUM/HIGH alerts to add to watchlist')
